# Phase 2: Chunk and Embed

This notebook:
1. Chunks all 284 transcripts with timestamp preservation
2. Embeds chunks using `nomic-embed-text` via Ollama
3. Stores in ChromaDB with episode metadata

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
from tqdm import tqdm

from config import TRANSCRIPTS_DIR, PROCESSED_DIR, CHROMA_DB_DIR, EMBEDDING_MODEL, CHUNK_SIZE, CHUNK_OVERLAP
from parser import parse_transcript, get_all_transcripts
from classifier import load_metadata
from chunker import chunk_transcript
from embedder import init_chroma_db, add_chunks_to_db

print("✓ Imports loaded successfully")

In [ ]:
# Load transcripts and test chunking
transcripts = list(get_all_transcripts(TRANSCRIPTS_DIR))
print(f"Found {len(transcripts)} transcripts")

# Test on first transcript
sample = parse_transcript(transcripts[0])
sample_chunks = chunk_transcript(sample, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

print(f"\nGuest: {sample.guest_name}")
print(f"Total segments: {len(sample.segments)}")
print(f"Total chunks: {len(sample_chunks)}")

print(f"\nFirst chunk:")
print(f"  Time: {sample_chunks[0].start_timestamp} - {sample_chunks[0].end_timestamp}")
print(f"  Tokens: ~{sample_chunks[0].token_count}")
print(f"  Speakers: {sample_chunks[0].speakers}")
print(f"  Preview: {sample_chunks[0].text[:300]}...")

Found 284 transcripts

Guest: Ada Chen Rekhi
Total segments: 202
Total chunks: 51

First chunk:
  Time: 00:00:00 - 00:01:21
  Tokens: ~467
  Speakers: ['Ada Chen Rekhi', 'Lenny']
  Preview: Ada Chen Rekhi: It's a terrible outcome to wake up one day and be late career and feel trapped because you have a certain lifestyle or a certain expectations of the people around you that you have to go work this job, but then you look at yourself in the mirror and you're not happy going in there. I...


In [ ]:
# Chunk all transcripts
all_chunks = []

for filepath in tqdm(transcripts, desc="Chunking transcripts"):
    parsed = parse_transcript(filepath)
    chunks = chunk_transcript(parsed, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    all_chunks.extend(chunks)

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunks per episode: {len(all_chunks) / len(transcripts):.1f}")

Chunking transcripts: 100%|██████████| 284/284 [00:00<00:00, 717.37it/s]


Total chunks created: 13014
Average chunks per episode: 45.8


In [ ]:
# Initialize ChromaDB
client, collection = init_chroma_db(CHROMA_DB_DIR)

print(f"ChromaDB path: {CHROMA_DB_DIR}")
print(f"Collection: {collection.name}")
print(f"\nExisting chunks in DB: {collection.count()}")

ChromaDB path: /Users/nanditakrishnan/llm/lenny-podcast/notebooks/../data/chroma_db
Collection: lenny_podcast_chunks

Existing chunks in DB: 1875


In [ ]:
# Load episode metadata for enrichment
METADATA_FILE = PROCESSED_DIR / "episodes_metadata.json"
metadata_list = load_metadata(str(METADATA_FILE))
metadata_dict = {m.guest_name: m for m in metadata_list}
print(f"Loaded metadata for {len(metadata_dict)} episodes")

# Add chunks to ChromaDB with progress
BATCH_SIZE = 25  # Process in batches
stats = {"added": 0, "skipped": 0, "errors": 0}

for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Embedding chunks"):
    batch = all_chunks[i:i + BATCH_SIZE]
    batch_stats = add_chunks_to_db(collection, batch, metadata_dict, EMBEDDING_MODEL)
    for k in stats:
        stats[k] += batch_stats[k]

print(f"\nEmbedding complete!")
print(f"  Added: {stats['added']}")
print(f"  Skipped (existing): {stats['skipped']}")
print(f"  Errors: {stats['errors']}")

Embedding chunks: 100%|██████████| 521/521 [23:24<00:00,  2.70s/it]


Embedding complete!
  Added: 11139
  Skipped (existing): 1875
  Errors: 0


In [ ]:
# Verify final collection stats
print(f"Final collection stats:")
print(f"  Total chunks: {collection.count()}")

# Sample a few to verify metadata
sample = collection.peek(limit=10)
unique_guests = set(m["guest_name"] for m in sample["metadatas"])
print(f"  Sample guests: {list(unique_guests)}")

Final collection stats:
  Total chunks: 13014
  Sample guests: ['Andy Johns', 'Aishwarya Naresh Reganti + Kiriti Badam', 'Adam Fishman', 'Anuj Rathi', 'Andrew Wilkinson', 'Adam Grenier', 'Albert Cheng', 'Alex Komoroske', 'Anton Osika', 'Alexander Embiricos']



Query: How to find product-market fit

1. Christopher Lochhead (01:11:48 - 01:11:48)
   Distance: 0.478
   Preview: Christopher Lochhead: Pretty simple way to determine or to distill product market fit. And what product market fit has come to mean is, we're a brew pub, you and I want to start a craft beer place lik...

2. Benjamin Lauzier (00:21:34 - 00:22:50)
   Distance: 0.535
   Preview: Benjamin Lauzier: Exactly. Yeah, completely.

Lenny Rachitsky: Awesome. You mentioned this idea of product market fit and the things change, pre-product market fit, post-product market fit. Classic qu...

3. Rahul Vohra (00:52:14 - 00:53:12)
   Distance: 0.592
   Preview: Rahul Vohra: Absolutely. You don't want to lose money doing this. We always made money doing onboarding to be clear, it's just that at a certain point the mass market, whether it for us it's enterpris...

Query: Hiring your first PM

1. Gokul Rajaram (00:25:25 - 00:26:46)
   Distance: 0.576
   Preview: Gokul Rajaram: And if you're